# Iteratively Optimized Light-GBM Model

**Model:** Light-GBM

In [ ]:
import os
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier as lgbc
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, average_precision_score, classification_report, confusion_matrix,
    precision_recall_curve
)
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but LGBMClassifier was fitted with feature names"
)

# hide optuna log output but let the warnings show
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Data Input

Taking input of the competition data

In [ ]:
# Set paths
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e9'
TRAIN_PATH = os.path.join(DATA_DIR, 'train.csv')
TEST_PATH = os.path.join(DATA_DIR, 'test.csv')
SUBMISSION_PATH = os.path.join(DATA_DIR, 'sample_submission.csv')

# Load data
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUBMISSION_PATH)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

## Processing the Raw Data

* Mapping the target feature _Yes_: 1 and _No_: 0
* Scaling the numerical features with `Standard Scaler`
* Using `One Hot Encoder` for categorical features

In [ ]:
# Data Preprocessing
# Target variable mapping
target_col = 'Will_Buy_EV'

if train[target_col].dtype == 'object':
    train[target_col] = train[target_col].map({'No': 0, 'Yes': 1})

# Separate features and target
X = train.drop(columns=['id', target_col])
y = train[target_col]
X_test = test.drop(columns=['id'])

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['str','object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical features:", categorical_cols)
print("Numerical features:", numerical_cols)

### Convert Categorical Features for LightGBM

Converting default `string` to `category` type for native LightGBM processing

In [ ]:
# convert default categorical features of dataframe to light-gbm native categorical handling dtype
for cat_feature in categorical_cols:
    train[cat_feature] = train[cat_feature].astype("category")
    test[cat_feature] = test[cat_feature].astype("category")

## 5 Fold CV

* Using the best parms from version 4
* 5 Fold CV is run where 5 lightgbm models are trained
* Each model predicts on the oof data
* The same is used to predict on the test data and aggregated to get the final prediction

In [ ]:
# best params - got in v4
best_params = {
    'max_depth': 3,
    'n_estimators': 2118,
    'learning_rate': 0.06447836392215553,
    'num_leaves': 6,
    'min_child_samples': 81,
    'subsample': 0.8156064390376033,
    'colsample_bytree': 0.8210130019038757,
    'reg_alpha': 1.3978327472570622e-05,
    'reg_lambda': 0.17099007769746272,
    'min_split_gain': 0.16970946366885054,
    'scale_pos_weight': 2.407193478675026,
}

In [ ]:
# Final 5 Fold CV
print("Starting 5-Fold Stratified Cross-Validation...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
oof_test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"Training Fold {fold + 1}/5...")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Define the model pipeline
    pipeline = Pipeline(steps=[
        ('classifier', lgbc(
            objective="binary",
            **best_params,
            random_state=42,
            n_jobs=-1,
            verbosity=-1
        ))
    ])
    
    pipeline.fit(X_train, y_train)
    oof_preds[val_idx] = pipeline.predict_proba(X_val)[:, 1]
    oof_test_preds += pipeline.predict_proba(X_test)[:, 1]/5

    # ROC-AUC score for each fold
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print(f"Fold {fold + 1} ROC-AUC: {fold_auc:.4f}")

print("\nCross-Validation complete!")

# calculating the validation metrics using 0.5 as the threshold value
y_pred_binary = (oof_preds >= 0.5).astype(int)

# calculate overall metrics
accuracy = accuracy_score(y, y_pred_binary)
precision = precision_score(y, y_pred_binary)
recall = recall_score(y, y_pred_binary)
f1 = f1_score(y, y_pred_binary)
roc_auc = roc_auc_score(y, oof_preds)
pr_auc = average_precision_score(y, oof_preds)

print("-" * 30)
print("Baseline Model (5-Fold OOF) Performance:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")
print("-" * 30)

print("\nClassification Report:")
print(classification_report(y, y_pred_binary))

print("Confusion Matrix:")
print(confusion_matrix(y, y_pred_binary))

## Precision Recall Curve

In [ ]:
# Precision Recall Curve
precision_curve, recall_curve, thresholds = precision_recall_curve(y, oof_preds)

plt.plot(
    recall_curve,
    precision_curve,
    label=f"LightGBM (PR-AUC = {pr_auc:.4f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid()
plt.show()

## Creating Final Submission

In [ ]:
# Create submission
submission = pd.DataFrame({
    'id': test['id'],
    target_col: oof_test_preds
})

submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
submission.head()